In [ ]:
import numpy as np
import pandas as pd
import re

In [ ]:
#import genotype data
genotype = pd.read_csv(
    "../data/raw/Pf8_drug_resistance_marker_genotypes.tsv",
    sep="\t"
)

In [ ]:
# viewing data
genotype

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],kelch13_349-726_ns_changes,mdr1_dup_call,pm2_dup_call
0,FP0008-C,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
1,FP0009-C,C,I,E,T,CVIET,T,H,I,S,...,S,N,F,Y,VD,D,T,NaN,0,0
2,FP0010-CW,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
3,FP0011-CW,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
4,FP0012-CW,C,I,E,T,CVIET,T,H,I,S,...,S,N,F,D,VD,D,T,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24404,SPT92049,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,-,VD,D,T,-,-1,-1
24405,SPT92054,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,D,VD,D,T,NaN,0,-1
24406,SPT92057,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,D,VD,D,T,-,-1,-1
24407,SPT94772,C,I,E,T,CVIET,T,H,I,S,...,-,-,-,-,VD,D,-,-,-1,-1


### Data exploration

In [ ]:
print(genotype["crt_76[K]"].value_counts(dropna=False))

crt_76[K]
T       12432
K       10768
T,K       607
K,T       550
-          48
T,Q         3
T,Q*        1
Name: count, dtype: int64


In [ ]:
# viewing all column count data
genotype_dic = {}
for col in genotype.columns:
    value_count = genotype[col].value_counts(dropna = False)
    genotype_dic[col] = value_count

In [ ]:
print(genotype_dic)

{'Sample': Sample
FP0008-C     1
FP0009-C     1
FP0010-CW    1
FP0011-CW    1
FP0012-CW    1
            ..
SPT92049     1
SPT92054     1
SPT92057     1
SPT94772     1
SPT94773     1
Name: count, Length: 24409, dtype: int64, 'crt_72[C]': crt_72[C]
C      23955
S        402
-         34
S,C        6
C,S        5
C,Y        5
Y          2
Name: count, dtype: int64, 'crt_74[M]': crt_74[M]
I      11883
M      11318
I,M      612
M,I      544
-         52
Name: count, dtype: int64, 'crt_75[N]': crt_75[N]
N       11205
E       11017
D         839
E,N       594
N,E       525
E,D        71
D,E        62
-          52
N,D        19
D,N         9
E,K         7
N,E*        3
E,N*        3
K           2
K,E         1
Name: count, dtype: int64, 'crt_76[K]': crt_76[K]
T       12432
K       10768
T,K       607
K,T       550
-          48
T,Q         3
T,Q*        1
Name: count, dtype: int64, 'crt_72-76[CVMNK]': crt_72-76[CVMNK]
CVIET           10898
CVMNK           10749
CVIDT             839
CVIET,CV

In [ ]:
genotype.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call'],
      dtype='str')

In [ ]:
# handling kelchi data
genotype['kelch13_349-726_ns_changes'].unique()

<StringArray>
[    nan,     '-', 'a676s', 'p441s', 'E612D', 'A676S', 'S522C', 'a578s',
 's522i', 'e509d',
 ...
 'v566i', 'p527s', 'a427v', 'v566l', 'd641n', 'S364Y', 'V487E', 's477y',
 'T350S', 'c469y']
Length: 266, dtype: str

## Classifying known biomarkers

In [ ]:
genotype.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call'],
      dtype='str')

In [ ]:
# creating a dataframe for known resistance biomakers
encoded = pd.DataFrame()

encoded["Sample"] = genotype["Sample"]

In [ ]:
# markers associated with drug resistance

associated_marker = {

    "dhfr_51[N]": {
        "gene": "dhfr",
        "wildtype": "N",
        "resistant": {"I"}
    },

    "dhfr_59[C]": {
        "gene": "dhfr",
        "wildtype": "C",
        "resistant": {"R"}
    },

    "dhfr_164[I]": {
        "gene": "dhfr",
        "wildtype": "I",
        "resistant": {"L"}
    },


    "dhps_540[K]": {
        "gene": "dhps",
        "wildtype": "K",
        "resistant": {"E"}
    },

    "dhps_581[A]": {
        "gene": "dhps",
        "wildtype": "A",
        "resistant": {"G"}
    },

    "dhps_613[A]": {
        "gene": "dhps",
        "wildtype": "A",
        "resistant": {"S", "T"}
    }
}

In [ ]:
marker_info = {
    "crt_76[K]": {
        "wildtype": "K",
        "resistant": {"T"}
    },

    "dhfr_108[S]": {
        "wildtype": "S",
        "resistant": {"N"}
    },

    "dhps_437[G]": {
        "wildtype": "A",
        "resistant": {"G"}
    }
}

In [ ]:
# known columns
known_marker = {**marker_info,**associated_marker }
known_encoder = []

for col, item in known_marker.items():
    known_encoder.append(col)


In [ ]:
def classify_known_marker(value, wildtype, resistance_alleles):

    if pd.isna(value):
        return "Missing"

    value = str(value).strip()

    if value in {".", "-", "*", "!"}:
        return "Missing"

    if "," in value:

        haps = [x.strip().upper() for x in value.split(",")]

        if any(x in {"", "-", "*", "!"} for x in haps):
            return "Missing"

        if any(x in resistance_alleles for x in haps):
            return "Resistant"

        if all(x == wildtype.upper() for x in haps):
            return "Sensitive"

        return "Missing"

    allele = value.upper()

    if allele in resistance_alleles:
        return "Resistant"

    if allele == wildtype.upper():
        return "Sensitive"

    return "Missing"

In [ ]:
# handle mutations causing drug resistance
for column, info in marker_info.items():

    encoded[column] = genotype[column].apply(
        lambda x: classify_known_marker(
            x,
            wildtype=info["wildtype"],
            resistance_alleles=info["resistant"]
        )
    )

In [ ]:
# handle mutations associated with resistance
for column, info in associated_marker.items():
    new_column = f"{column}_assoc"
    encoded[new_column] = genotype[column].apply(
        lambda x: classify_known_marker(
            x,
            wildtype=info["wildtype"],
            resistance_alleles=info["resistant"]
        )
    )

In [ ]:
encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive


#### Handling kelch13

In [ ]:
kelch13_col = "kelch13_349-726_ns_changes"

In [ ]:
k13_resistance = {
    "F446I",
    "N458Y",
    "C469Y",
    "M476I",
    "Y493H",
    "R539T",
    "I543T",
    "P553L",
    "R561H",
    "P574L",
    "C580Y",
    "R622I",
    "A675V"
}

In [ ]:
# classifying k13
def classify_k13(value):

    if pd.isna(value):
        return "Missing"

    value = str(value).strip()

    if value in {"", ".", "-", "*", "!"}:
        return "Missing"

    # Some cells may contain multiple mutations
    mutations = [
        x.strip().upper()
        for x in value.split(",")
    ]

    if any(m in k13_resistance for m in mutations):
        return "Resistant"

    return "Other"

In [ ]:
encoded["kelch13_known"] = genotype[
    "kelch13_349-726_ns_changes"
].apply(classify_k13)

In [ ]:
encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,kelch13_known
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Missing
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing


#### Handling resistance due to multiple column copy

In [ ]:
# encoding if multiple copies associated with drug resistance
def copy_encoder(value):

    # Missing
    if pd.isna(value) or str(value).strip() == "":
        return "Missing"

    value = str(value).strip()

    # ---------- Special: copy number ----------
    # ----- Special: copy number ----------
    
    if int(value) == 0:
        return "Sensitive"
    elif int(value) == 1:
        return "Resistant"
    elif value in {"-", "*", "!"}:
        return "missing"
    else:

        return "rare"


In [ ]:
encoded[["mdr1_dup_call", "pm2_dup_call"]] = genotype[["mdr1_dup_call", "pm2_dup_call"]].map(copy_encoder)


In [ ]:
encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,kelch13_known,mdr1_dup_call,pm2_dup_call
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive


In [ ]:
encoded.to_csv('Important_genotype.csv',index = True)

## Handling other columns

In [ ]:
genotype_col = genotype.columns.to_list()
#known_marker.append(["mdr1_dup_call", "pm2_dup_call","kelch13_349-726_ns_changes"])
len(genotype_col)
genotype_col


['Sample',
 'crt_72[C]',
 'crt_74[M]',
 'crt_75[N]',
 'crt_76[K]',
 'crt_72-76[CVMNK]',
 'crt_93[T]',
 'crt_97[H]',
 'crt_218[I]',
 'crt_220[A]',
 'crt_271[Q]',
 'crt_326[N]',
 'crt_333[T]',
 'crt_353[G]',
 'crt_356[I]',
 'crt_371[R]',
 'dhfr_16[N]',
 'dhfr_51[N]',
 'dhfr_59[C]',
 'dhfr_108[S]',
 'dhfr_164[I]',
 'dhfr_306[S]',
 'dhps_436[S]',
 'dhps_437[G]',
 'dhps_540[K]',
 'dhps_581[A]',
 'dhps_613[A]',
 'exo_415[E]',
 'mdr1_86[N]',
 'mdr1_184[Y]',
 'mdr1_1034[S]',
 'mdr1_1042[N]',
 'mdr1_1226[F]',
 'mdr1_1246[D]',
 'arps10_127-128[VD]',
 'fd_193[D]',
 'mdr2_484[T]',
 'kelch13_349-726_ns_changes',
 'mdr1_dup_call',
 'pm2_dup_call']

In [ ]:
rare_col = [
    col for col in genotype.columns
    if col not in known_marker
    and col != "Sample"
]

In [ ]:
exclude = {
    "Sample",
    "crt_72-76[CVMNK]",
    "kelch13_349-726_ns_changes",
    "mdr1_dup_call",
    "pm2_dup_call"
}

rare_col = [
    col for col in genotype.columns
    if col not in known_marker
    and col not in exclude
]

print(rare_col)

['crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]', 'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]', 'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_306[S]', 'dhps_436[S]', 'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]', 'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]', 'fd_193[D]', 'mdr2_484[T]']


### Handling rare mutations

In [ ]:
# extracting rfference ammino acide for rare columns
import re

rare_ref = {}

for col in rare_col:
    match = re.search(r'\[([A-Z]+)\]', col)
    
    if match:
        rare_ref[col] = match.group(1)

print(rare_ref)

{'crt_72[C]': 'C', 'crt_74[M]': 'M', 'crt_75[N]': 'N', 'crt_93[T]': 'T', 'crt_97[H]': 'H', 'crt_218[I]': 'I', 'crt_220[A]': 'A', 'crt_271[Q]': 'Q', 'crt_326[N]': 'N', 'crt_333[T]': 'T', 'crt_353[G]': 'G', 'crt_356[I]': 'I', 'crt_371[R]': 'R', 'dhfr_16[N]': 'N', 'dhfr_306[S]': 'S', 'dhps_436[S]': 'S', 'exo_415[E]': 'E', 'mdr1_86[N]': 'N', 'mdr1_184[Y]': 'Y', 'mdr1_1034[S]': 'S', 'mdr1_1042[N]': 'N', 'mdr1_1226[F]': 'F', 'mdr1_1246[D]': 'D', 'arps10_127-128[VD]': 'VD', 'fd_193[D]': 'D', 'mdr2_484[T]': 'T'}


In [ ]:
# encoding rare (undetermined column)
def encode_rare(value, ref):
    
    # Missing value
    if pd.isna(value):
        return "Missing"
    
    value = str(value).strip()
    
    # Missing/uncertain symbols
    if value == "" or any(x in value for x in ["*", "!", "."]):
        return "Missing"
    
    # Split heterozygous calls
    alleles = [x.strip() for x in value.split(",")]
    
    # If ALL observed alleles are reference
    if all(allele == ref for allele in alleles):
        return "sensitive"
    
    # Any non-reference mutation
    return "Rare"

In [ ]:


for col in rare_col:
    
    ref = rare_ref[col]
    
    encoded[col] = genotype[col].apply(
        lambda x: encode_rare(x, ref)
    )

In [ ]:
encoded

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,exo_415[E],mdr1_86[N],mdr1_184[Y],mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T]
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,Rare,Rare,sensitive,sensitive,sensitive,Rare,sensitive,sensitive,sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,Rare,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,Rare,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24404,SPT92049,Sensitive,Resistant,Missing,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,Rare,sensitive,sensitive,sensitive
24405,SPT92054,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
24406,SPT92057,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
24407,SPT94772,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,Rare,Rare,Rare,Rare,sensitive,sensitive,Rare


In [ ]:
encoded.shape

(24409, 39)

# Handling the drug phenotype

In [ ]:
# loading sample metadata
metadata = pd.read_csv("../data/raw/Pf8-samples.csv")

In [ ]:
metadata.columns

Index(['sample_id', 'year', 'qc_pass', 'study_id', 'region', 'country',
       'country_id', 'site', 'site_id', 'ARTresistant', 'CQresistant',
       'MQresistant', 'PPQresistant', 'PYRresistant', 'SDXresistant'],
      dtype='str')

In [ ]:
# dropping unnecessary columns
del_col = ['year', 'qc_pass', 'study_id', 'region', 'country', 'country_id', 'site', 'site_id']

In [ ]:
merged_encoded = encoded.merge(metadata, left_on = 'Sample', right_on = 'sample_id', how = 'left')

In [ ]:
merged_encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,country,country_id,site,site_id,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant,SDXresistant
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,sensitive,sensitive,undetermined,undetermined
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,sensitive,sensitive,resistant,sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,resistant
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,undetermined
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,undetermined,undetermined,resistant,sensitive


In [ ]:
merged_encoded.shape

(24409, 54)

In [ ]:
# removing unnecessary columns 
merged_encoded.drop(columns = del_col, inplace = True)

In [ ]:
merged_encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,arps10_127-128[VD],fd_193[D],mdr2_484[T],sample_id,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant,SDXresistant
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0008-C,sensitive,undetermined,sensitive,sensitive,undetermined,undetermined
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0009-C,sensitive,resistant,sensitive,sensitive,resistant,sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0010-CW,sensitive,undetermined,undetermined,undetermined,resistant,resistant
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0011-CW,sensitive,undetermined,undetermined,undetermined,resistant,undetermined
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0012-CW,sensitive,resistant,undetermined,undetermined,resistant,sensitive


In [ ]:
merged_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 24409 entries, 0 to 24408
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Sample              24409 non-null  str  
 1   crt_76[K]           24409 non-null  str  
 2   dhfr_108[S]         24409 non-null  str  
 3   dhps_437[G]         24409 non-null  str  
 4   dhfr_51[N]_assoc    24409 non-null  str  
 5   dhfr_59[C]_assoc    24409 non-null  str  
 6   dhfr_164[I]_assoc   24409 non-null  str  
 7   dhps_540[K]_assoc   24409 non-null  str  
 8   dhps_581[A]_assoc   24409 non-null  str  
 9   dhps_613[A]_assoc   24409 non-null  str  
 10  kelch13_known       24409 non-null  str  
 11  mdr1_dup_call       24409 non-null  str  
 12  pm2_dup_call        24409 non-null  str  
 13  crt_72[C]           24409 non-null  str  
 14  crt_74[M]           24409 non-null  str  
 15  crt_75[N]           24409 non-null  str  
 16  crt_93[T]           24409 non-null  str  
 17  crt_

## Handling target columns

In [ ]:
target_df = merged_encoded[['ARTresistant', 'CQresistant',
       'MQresistant', 'PPQresistant', 'PYRresistant', 'SDXresistant']]

In [ ]:
target_col = ['ARTresistant', 'CQresistant','MQresistant', 'PPQresistant', 'PYRresistant', 'SDXresistant']

In [ ]:
pd.set_option('display.max_rows', None)
for col in target_df.columns:
    print(f"\n{col}")
    print(target_df[col].value_counts(dropna=False))


ARTresistant
ARTresistant
sensitive       17147
resistant        3905
undetermined     3357
Name: count, dtype: int64

CQresistant
CQresistant
resistant       12432
sensitive       10768
undetermined     1209
Name: count, dtype: int64

MQresistant
MQresistant
sensitive       12598
undetermined    11153
resistant         658
Name: count, dtype: int64

PPQresistant
PPQresistant
sensitive       13467
undetermined    10454
resistant         488
Name: count, dtype: int64

PYRresistant
PYRresistant
resistant       21964
undetermined     1230
sensitive        1215
Name: count, dtype: int64

SDXresistant
SDXresistant
resistant       19270
sensitive        3601
undetermined     1538
Name: count, dtype: int64


### Encoding the y_target column

In [ ]:
import numpy as np

y = merged_encoded[target_col].replace({
    "sensitive": 0,
    "resistant": 1,
    "undetermined": 0
})

In [ ]:
y.head()

,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant,SDXresistant
0,0,0,0,0,0,0
1,0,1,0,0,1,0
2,0,0,0,0,1,1
3,0,0,0,0,1,0
4,0,1,0,0,1,0


## Encoding features variable

In [ ]:
features_df = merged_encoded.drop(columns = target_col)

In [ ]:
features_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24409 entries, 0 to 24408
Data columns (total 40 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Sample              24409 non-null  str  
 1   crt_76[K]           24409 non-null  str  
 2   dhfr_108[S]         24409 non-null  str  
 3   dhps_437[G]         24409 non-null  str  
 4   dhfr_51[N]_assoc    24409 non-null  str  
 5   dhfr_59[C]_assoc    24409 non-null  str  
 6   dhfr_164[I]_assoc   24409 non-null  str  
 7   dhps_540[K]_assoc   24409 non-null  str  
 8   dhps_581[A]_assoc   24409 non-null  str  
 9   dhps_613[A]_assoc   24409 non-null  str  
 10  kelch13_known       24409 non-null  str  
 11  mdr1_dup_call       24409 non-null  str  
 12  pm2_dup_call        24409 non-null  str  
 13  crt_72[C]           24409 non-null  str  
 14  crt_74[M]           24409 non-null  str  
 15  crt_75[N]           24409 non-null  str  
 16  crt_93[T]           24409 non-null  str  
 17  crt_

In [ ]:
features_df.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,mdr1_86[N],mdr1_184[Y],mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],sample_id
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,FP0008-C
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,Rare,sensitive,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,FP0009-C
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,FP0010-CW
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,FP0011-CW
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,FP0012-CW


In [ ]:
features_df = features_df.drop(
    columns=["Sample", "sample_id"],
    errors="ignore"
).copy()

In [ ]:
for col in features_df.columns:
    print(f"\n===== {col} =====")
    print(features_df[col].value_counts(dropna=False))


===== crt_76[K] =====
crt_76[K]
Resistant    13593
Sensitive    10768
Missing         48
Name: count, dtype: int64

===== dhfr_108[S] =====
dhfr_108[S]
Resistant    22780
Sensitive     1215
Missing        414
Name: count, dtype: int64

===== dhps_437[G] =====
dhps_437[G]
Resistant    20538
Sensitive     3601
Missing        270
Name: count, dtype: int64

===== dhfr_51[N]_assoc =====
dhfr_51[N]_assoc
Resistant    20083
Sensitive     4029
Missing        297
Name: count, dtype: int64

===== dhfr_59[C]_assoc =====
dhfr_59[C]_assoc
Resistant    22051
Sensitive     2034
Missing        323
Other            1
Name: count, dtype: int64

===== dhfr_164[I]_assoc =====
dhfr_164[I]_assoc
Sensitive    19106
Resistant     5032
Missing        269
Other            2
Name: count, dtype: int64

===== dhps_540[K]_assoc =====
dhps_540[K]_assoc
Sensitive    12850
Resistant     8386
Other         2810
Missing        363
Name: count, dtype: int64

===== dhps_581[A]_assoc =====
dhps_581[A]_assoc
Sensitive    1

In [ ]:
features_df = features_df[
    features_df['dhps_613[A]_assoc'] != "Other"
].copy()

In [ ]:
for col in features_df.columns:
    print(f"\n===== {col} =====")
    print(features_df[col].value_counts(dropna=False))


===== crt_76[K] =====
crt_76[K]
Resistant    13593
Sensitive    10767
Missing         48
Name: count, dtype: int64

===== dhfr_108[S] =====
dhfr_108[S]
Resistant    22779
Sensitive     1215
Missing        414
Name: count, dtype: int64

===== dhps_437[G] =====
dhps_437[G]
Resistant    20538
Sensitive     3600
Missing        270
Name: count, dtype: int64

===== dhfr_51[N]_assoc =====
dhfr_51[N]_assoc
Resistant    20082
Sensitive     4029
Missing        297
Name: count, dtype: int64

===== dhfr_59[C]_assoc =====
dhfr_59[C]_assoc
Resistant    22050
Sensitive     2034
Missing        323
Other            1
Name: count, dtype: int64

===== dhfr_164[I]_assoc =====
dhfr_164[I]_assoc
Sensitive    19105
Resistant     5032
Missing        269
Other            2
Name: count, dtype: int64

===== dhps_540[K]_assoc =====
dhps_540[K]_assoc
Sensitive    12849
Resistant     8386
Other         2810
Missing        363
Name: count, dtype: int64

===== dhps_581[A]_assoc =====
dhps_581[A]_assoc
Sensitive    1

In [ ]:
features_df['crt_371[R]'].value_counts()

crt_371[R]
Rare         12455
sensitive    11953
Name: count, dtype: int64

### Encoding the X features

In [ ]:
# defining different columns 
known_cols = list(marker_info.keys())
known_cols.append("mdr1_dup_call")
known_cols.append("pm2_dup_call")


associated_cols = list(associated_marker.keys())

rare_cols = rare_col

In [ ]:
print("Known:", known_cols)
print("Associated:", associated_cols)
print("Rare:", rare_cols)

Known: ['crt_76[K]', 'dhfr_108[S]', 'dhps_437[G]', 'mdr1_dup_call', 'pm2_dup_call']
Associated: ['dhfr_51[N]', 'dhfr_59[C]', 'dhfr_164[I]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]']
Rare: ['crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]', 'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]', 'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_306[S]', 'dhps_436[S]', 'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]', 'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]', 'fd_193[D]', 'mdr2_484[T]']


In [ ]:
print([c for c in features_df.columns if 'dhfr' in c or 'dhps' in c])

['dhfr_108[S]', 'dhps_437[G]', 'dhfr_51[N]_assoc', 'dhfr_59[C]_assoc', 'dhfr_164[I]_assoc', 'dhps_540[K]_assoc', 'dhps_581[A]_assoc', 'dhps_613[A]_assoc', 'dhfr_16[N]', 'dhfr_306[S]', 'dhps_436[S]']


In [ ]:
assoc_col = [c for c in encoded.columns if '_assoc' in c]

In [ ]:
# encoding known and associated columns

X_known = features_df[known_cols].replace({
    "Resistant": 10,
    "Sensitive": 2,
    "sensitive": 2,
    "Missing": 0,
    "rare": 3
})

X_associated = features_df[assoc_col].replace({
    "Resistant": 5,
    "Sensitive": 2,
    "sensitive": 2,
    "Missing": 0,
    "rare": 3
})

In [ ]:
# viewing columns associated with known resistance
for col in known_cols:
    print(col, X_known[col].value_counts(dropna=False))

crt_76[K] crt_76[K]
10    13593
2     10767
0        48
Name: count, dtype: int64
dhfr_108[S] dhfr_108[S]
10    22779
2      1215
0       414
Name: count, dtype: int64
dhps_437[G] dhps_437[G]
10    20538
2      3600
0       270
Name: count, dtype: int64
mdr1_dup_call mdr1_dup_call
2     19511
3      4195
10      702
Name: count, dtype: int64
pm2_dup_call pm2_dup_call
2     17610
3      4718
10     2080
Name: count, dtype: int64


In [ ]:
# viewing columns assocciated with resistance associations
for col in assoc_col:
    print(col, X_associated[col].value_counts(dropna=False))

dhfr_51[N]_assoc dhfr_51[N]_assoc
5    20082
2     4029
0      297
Name: count, dtype: int64
dhfr_59[C]_assoc dhfr_59[C]_assoc
5        22050
2         2034
0          323
Other        1
Name: count, dtype: int64
dhfr_164[I]_assoc dhfr_164[I]_assoc
2        19105
5         5032
0          269
Other        2
Name: count, dtype: int64
dhps_540[K]_assoc dhps_540[K]_assoc
2        12849
5         8386
Other     2810
0          363
Name: count, dtype: int64
dhps_581[A]_assoc dhps_581[A]_assoc
2    18694
5     5349
0      365
Name: count, dtype: int64
dhps_613[A]_assoc dhps_613[A]_assoc
2    22744
5     1359
0      305
Name: count, dtype: int64


In [ ]:
# Handling rare mutations
X_rare = features_df[rare_cols].replace({
    "sensitive": 0,
    "Sensitive": 0,
    "Rare": 1,
    "Missing": np.nan
})

## Haplotypes associated with drug resistance

In [ ]:
# columns for mutations in Sulfadoxine-Pyrimethamine (treatment) and Sulfadoxine-Pyrimethamine (IPTp)
dhfr_cols = [
    "dhfr_51[N]",
    "dhfr_59[C]",
    "dhfr_108[S]",
    "dhfr_164[I]"
]

dhps_cols = [
    "dhps_437[G]",
    "dhps_540[K]",
    "dhps_581[A]",
    "dhps_613[A]"
]

combo_cols = dhfr_cols + dhps_cols

In [ ]:
# dealing with SP-IPTp (sextuple mutant)
def classify_sp(row):

    cols = [
        "dhfr_51[N]",
        "dhfr_59[C]",
        "dhfr_108[S]",
        "dhfr_164[I]",
        "dhps_437[G]",
        "dhps_540[K]",
        "dhps_581[A]",
        "dhps_613[A]"
    ]

    if any(is_ambiguous(row[c]) for c in cols):
        return "Rare"

    if (
        row["dhfr_51[N]"] == "I"
        and row["dhfr_59[C]"] == "R"
        and row["dhfr_108[S]"] == "N"
        and row["dhps_437[G]"] == "G"
        and row["dhps_540[K]"] == "E"
        and (
            row["dhfr_164[I]"] == "L"
            or row["dhps_581[A]"] == "G"
            or row["dhps_613[A]"] in {"S", "T"}
        )
    ):
        return "Resistant"

    return "Sensitive"

In [ ]:
# helper function
def is_ambiguous(value):

    if pd.isna(value):
        return True

    value = str(value).strip()

    if value in {"-", "*", "!"}:
        return True

    return value.islower()

In [ ]:
# dealing with Pyrimethamine (triple mutant)
def classify_pyr(row):

    cols = [
        "dhfr_51[N]",
        "dhfr_59[C]",
        "dhfr_108[S]"
    ]

    if any(is_ambiguous(row[c]) for c in cols):
        return "Rare"

    if (
        row["dhfr_51[N]"] == "I"
        and row["dhfr_59[C]"] == "R"
        and row["dhfr_108[S]"] == "N"
    ):
        return "Resistant"

    return "Sensitive"

In [ ]:
# pyresistant is for triplet Pyrimethamine
encoded["PYRresistant"] = genotype.apply(classify_pyr, axis=1)
# Spresistant if for sextuple mutant
encoded["SPresistant"] = genotype.apply(classify_sp, axis=1)

In [ ]:
encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,mdr1_184[Y],mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],PYRresistant,SPresistant
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,Sensitive,Sensitive
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,sensitive,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,Resistant,Sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,Resistant,Sensitive
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,Resistant,Sensitive
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,Resistant,Sensitive


In [ ]:
def encode(value, column):

    # Missing
    if pd.isna(value) or str(value).strip() == "":
        return "Missing"

    value = str(value).strip()

    # ---------- Special: copy number ----------
    # ----- Special: copy number ----------
    if column in ["mdr1_dup_call", "pm2_dup_call"]:
        if int(value) == 0:
            return "Sensitive"
        elif int(value) == 1:
            return "Resistant"
        elif int(value) in {"-", "*", "!"}:
            return "missing"
        else:
            return "rare"
    

    # ---------- Special: crt haplotype ----------
    if column == "crt_72-76[CVMNK]":

        haps = [x.strip() for x in value.split(",")]

        if any(h in {"CVIET", "SVMNT"} for h in haps):
            return "Resistant"

        if all(h == "CVMNK" for h in haps):
            return "Sensitive"

        return "Rare"

    # ---------- Special: kelch13 ----------
    if column == "kelch13_349-726_ns_changes":

        # Missing or unresolved
        if pd.isna(value):
            return "missing"

        value = str(value).strip()

        if value in {"-", "*", "!"}:
            return "missing"

        # Two haplotypes
        if "," in value:
            haps = [x.strip() for x in value.split(",")]

            # Same mutation on both haplotypes
            if len(set(haps)) == 1:
                value = haps[0]
            else:
                return "Rare"

        # Lowercase = heterozygous
        if value.islower():
            return "Rare"

        value = value.upper()

        # Wild type (if present in your data)
        if value in {"WT", "578S"}:
            return "Sensitive"

        # WHO resistance mutation
        if value in WHO_K13:
            return "Resistant"

        # Any other homozygous mutation
        return "Sensitive"
    
    

    # ---------- Multiple alleles ----------
    alleles = [a.strip() for a in value.split(",")]
    if len(alleles) > 1:

        if column in SPECIAL:
            if any(a in SPECIAL[column] for a in alleles):
                return "Resistant"
            return "Rare"

        # General columns
        if ref_aa[column] in alleles:
            return "Sensitive"

        return "Rare"

    # ---------- Single allele ----------

    allele = alleles[0]

    if column in SPECIAL:
        if allele in SPECIAL[column]:
            return "rare"

    if allele == ref_aa[column]:
        return "Sensitive"

    return "missing"

### Handling artesunate_mefloquine

In [ ]:
def classify_artesunate_mefloquine(row):

    art = row["kelch13_349-726_ns_changes"]
    mq = row["mdr1_dup_call"]

    # Missing
    if art == "Missing" or mq == "Missing":
        return "Missing"

    # Both resistant
    if art == "Resistant" and mq == "Resistant":
        return "Resistant"

    # At least one sensitive
    if art == "Sensitive" or mq == "Sensitive":
        return "Sensitive"

    # Rare/unknown combinations
    return "Rare"

In [ ]:
encoded["ASMQresistant"] = encoded.apply(
    classify_artesunate_mefloquine,
    axis=1
)

KeyError: 'kelch13_349-726_ns_changes'

### Handling piperaquine

In [ ]:
encoded.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call', 'PYRresistant', 'SPresistant',
       'ASMQresistant'],
      dtype='str')

In [ ]:
def classify_artesunate_mefloquine(row):

    art = row["kelch13_349-726_ns_changes"]
    mq = row["mdr1_dup_call"]

    # Missing
    if art == "Missing" or mq == "Missing":
        return "Missing"

    # Both resistant
    if art == "Resistant" and mq == "Resistant":
        return "Resistant"

    # At least one sensitive
    if art == "Sensitive" or mq == "Sensitive":
        return "Sensitive"

    # Rare/unknown combinations
    return "Rare"

In [ ]:
encoded

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],kelch13_349-726_ns_changes,mdr1_dup_call,pm2_dup_call,PYRresistant,SPresistant,ASMQresistant
0,FP0008-C,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Sensitive,Sensitive,Missing
1,FP0009-C,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,missing,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
2,FP0010-CW,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
3,FP0011-CW,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
4,FP0012-CW,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24404,SPT92049,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,missing,Sensitive,Sensitive,Sensitive,missing,rare,rare,Resistant,Rare,Rare
24405,SPT92054,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,rare,Resistant,Sensitive,Missing
24406,SPT92057,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,missing,rare,rare,Resistant,Sensitive,Rare
24407,SPT94772,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,missing,Sensitive,Sensitive,missing,missing,rare,rare,Resistant,Sensitive,Rare


## Filtering African countries only

,sample_id,year,qc_pass,study_id,region,country,country_id,site,site_id,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant,SDXresistant
0,FP0008-C,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,sensitive,sensitive,undetermined,undetermined
1,FP0009-C,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,sensitive,sensitive,resistant,sensitive
2,FP0010-CW,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,resistant
3,FP0011-CW,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,undetermined
4,FP0012-CW,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,undetermined,undetermined,resistant,sensitive


In [ ]:
metadata['ARTresistant'].unique()

<StringArray>
['sensitive', nan, 'undetermined', 'resistant']
Length: 4, dtype: str

In [ ]:
metadata['ARTresistant'].value_counts()

ARTresistant
sensitive       17147
resistant        3905
undetermined     3357
Name: count, dtype: int64

In [ ]:
metadata['ARTresistant'].isna().sum()

np.int64(8586)

In [ ]:
metadata['country'].unique()

<StringArray>
[                      'Mauritania',                           'Gambia',
                           'Guinea',                            'Kenya',
                         'Thailand',                         'Tanzania',
                            'Ghana',                         'Cambodia',
                        'Indonesia',                     'Burkina Faso',
                             'Mali',                 'Papua New Guinea',
                             'Peru',                       'Bangladesh',
                           'Malawi',                          'Vietnam',
                         'Colombia',                        'Venezuela',
                           'Uganda',                          'Myanmar',
                             'Laos', 'Democratic Republic of the Congo',
                          'Nigeria',                       'Madagascar',
                         'Cameroon',                    'Côte d'Ivoire',
                         'Ethiopia', 

<class 'pandas.DataFrame'>
Index: 20951 entries, 0 to 32994
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sample_id     20951 non-null  str    
 1   year          20951 non-null  int64  
 2   qc_pass       20951 non-null  bool   
 3   study_id      20951 non-null  int64  
 4   region        0 non-null      float64
 5   country       20951 non-null  str    
 6   country_id    20951 non-null  str    
 7   site          20951 non-null  str    
 8   site_id       20951 non-null  str    
 9   ARTresistant  14508 non-null  str    
 10  CQresistant   14508 non-null  str    
 11  MQresistant   14508 non-null  str    
 12  PPQresistant  14508 non-null  str    
 13  PYRresistant  14508 non-null  str    
 14  SDXresistant  14508 non-null  str    
dtypes: bool(1), float64(1), int64(2), str(11)
memory usage: 2.4 MB


In [ ]:
africa_df.columns

Index(['sample_id', 'year', 'qc_pass', 'study_id', 'country', 'country_id',
       'site', 'site_id', 'ARTresistant', 'CQresistant', 'MQresistant',
       'PPQresistant', 'PYRresistant', 'SDXresistant'],
      dtype='str')

In [ ]:
print(len(african_countries_geno))

20951


In [ ]:
african_countries_geno.head()

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,country,country_id,site,site_id,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant_y,SDXresistant
0,FP0008-C,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,sensitive,sensitive,undetermined,undetermined
1,FP0009-C,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,sensitive,sensitive,resistant,sensitive
2,FP0010-CW,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,resistant
3,FP0011-CW,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,undetermined
4,FP0012-CW,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,undetermined,undetermined,resistant,sensitive


In [ ]:
african_countries_geno.info()

<class 'pandas.DataFrame'>
RangeIndex: 20951 entries, 0 to 20950
Data columns (total 57 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Sample                      14508 non-null  str  
 1   crt_72[C]                   14508 non-null  str  
 2   crt_74[M]                   14508 non-null  str  
 3   crt_75[N]                   14508 non-null  str  
 4   crt_76[K]                   14508 non-null  str  
 5   crt_72-76[CVMNK]            14508 non-null  str  
 6   crt_93[T]                   14508 non-null  str  
 7   crt_97[H]                   14508 non-null  str  
 8   crt_218[I]                  14508 non-null  str  
 9   crt_220[A]                  14508 non-null  str  
 10  crt_271[Q]                  14508 non-null  str  
 11  crt_326[N]                  14508 non-null  str  
 12  crt_333[T]                  14508 non-null  str  
 13  crt_353[G]                  14508 non-null  str  
 14  crt_356[I]       

# Merging with East African Countries

In [ ]:
# loading east africa data 
east_genotype = pd.read_csv('../processed/east-country_year2010-2019.csv')

In [ ]:
east_genotype.columns

Index(['Unnamed: 0', 'sample_id', 'year', 'qc_pass', 'study_id', 'region',
       'country', 'country_id', 'site', 'site_id', 'ARTresistant',
       'CQresistant', 'MQresistant', 'PPQresistant', 'PYRresistant',
       'SDXresistant'],
      dtype='str')

In [ ]:
# saving East africa countries encoded genotype data
east_african_geno.to_csv('../processed/East_africa_geno.csv')

# Merging with West African Countries